# Modified TypiClust Algorithm
Best configuration: UMAP (n=5) + Hybrid Selection (alpha=0.2) on CIFAR-10 with B=10.

## 1. Setup & Data Loading

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import umap

In [2]:
# Load CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=256, shuffle=False, num_workers=2)
testloader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

## 2. SimCLR Encoder (Representation Learning)

In [3]:
class ResNet18Encoder(nn.Module):
    """SimCLR encoder with ResNet-18 backbone adapted for CIFAR-10 (32x32)."""
    def __init__(self):
        super().__init__()
        self.resnet = models.resnet18(weights=None)
        self.resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.resnet.maxpool = nn.Identity()
        self.resnet.fc = nn.Identity()

    def forward(self, x):
        return self.resnet(x)


def load_encoder(weights_path, device='cuda'):
    encoder = ResNet18Encoder()
    encoder.load_state_dict(torch.load(weights_path, map_location=device))
    encoder = encoder.to(device)
    encoder.eval()
    return encoder


def extract_embeddings(encoder, dataloader, device='cuda'):
    encoder.eval()
    all_embeddings = []
    all_labels = []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            h = encoder(images)
            all_embeddings.append(h.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.vstack(all_embeddings), np.hstack(all_labels)

In [4]:
# Load pretrained SimCLR encoder and extract embeddings
encoder = load_encoder('./simclr_encoder_200ep.pth')
embeddings, labels = extract_embeddings(encoder, trainloader)
test_embeddings, test_labels = extract_embeddings(encoder, testloader)
print(f"Train embeddings: {embeddings.shape}")
print(f"Test embeddings: {test_embeddings.shape}")

Train embeddings: (50000, 512)
Test embeddings: (10000, 512)


## 3. UMAP Dimensionality Reduction (Modification 1)

In [5]:
def preprocess_umap(embeddings, n_components=5, random_state=42):
    """Reduce embedding dimensionality with UMAP before clustering."""
    reducer = umap.UMAP(n_components=n_components, random_state=random_state)
    transformed = reducer.fit_transform(embeddings)
    return transformed, reducer


# Apply UMAP (n=5) to train and test embeddings
proc_embeddings, umap_model = preprocess_umap(embeddings, n_components=5)
proc_test = umap_model.transform(test_embeddings)
print(f"UMAP train embeddings: {proc_embeddings.shape}")
print(f"UMAP test embeddings: {proc_test.shape}")

/mnt/data/conda/envs/ml-cw2/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP train embeddings: (50000, 5)
UMAP test embeddings: (10000, 5)


## 4. Typicality Computation

In [6]:
def compute_typicality(embeddings, k=20):
    """Compute typicality as inverse mean distance to k nearest neighbours."""
    nn = NearestNeighbors(n_neighbors=k + 1)
    nn.fit(embeddings)
    distances, _ = nn.kneighbors(embeddings)
    distances = distances[:, 1:]  # exclude self
    return np.mean(distances, axis=1) ** -1


typicality = compute_typicality(proc_embeddings, k=20)
print(f"Typicality shape: {typicality.shape}")
print(f"Typicality range: [{typicality.min():.4f}, {typicality.max():.4f}]")

Typicality shape: (50000,)
Typicality range: [1.4265, 49.2716]


## 5. Clustering (K-Means, K=B)

In [7]:
def cluster_standard(embeddings, budget, random_state=42):
    """K-Means clustering with K = budget."""
    kmeans = KMeans(n_clusters=budget, random_state=random_state, n_init=10)
    kmeans.fit(embeddings)
    return kmeans.labels_


budget = 10
cluster_labels = cluster_standard(proc_embeddings, budget)
print(f"Cluster sizes: {np.bincount(cluster_labels)}")

Cluster sizes: [5585 7169 4816 3861 4940 2399 4925 5329 6171 4805]


## 6. Hybrid Selection (Modification 2)

In [8]:
def select_hybrid(embeddings, typicality, cluster_labels, budget, alpha=0.2):
    """
    Hybrid selection: score = alpha * typicality + (1-alpha) * min_distance.
    Selects sequentially, balancing representativeness with spatial diversity.
    """
    selected = []

    # Order clusters by max typicality (most typical cluster first)
    cluster_order = []
    for c in range(budget):
        cluster_idx = np.where(cluster_labels == c)[0]
        cluster_order.append((c, np.max(typicality[cluster_idx])))
    cluster_order.sort(key=lambda x: x[1], reverse=True)

    # First point: most typical overall
    first_cluster = cluster_order[0][0]
    cluster_idx = np.where(cluster_labels == first_cluster)[0]
    best = cluster_idx[np.argmax(typicality[cluster_idx])]
    selected.append(best)

    # Subsequent points: hybrid score within each cluster
    for cluster in cluster_order[1:]:
        cluster_idx = np.where(cluster_labels == cluster[0])[0]
        candidate_embeddings = embeddings[cluster_idx]

        # Compute min distance to already-selected points
        dist = np.linalg.norm(
            candidate_embeddings[:, None] - embeddings[selected], axis=2
        )
        min_distance = dist.min(axis=1)

        # Normalise both terms to [0, 1] within this cluster
        norm_distance = (min_distance - min_distance.min()) / (
            min_distance.max() - min_distance.min() + 1e-8
        )
        norm_typicality = (typicality[cluster_idx] - typicality[cluster_idx].min()) / (
            typicality[cluster_idx].max() - typicality[cluster_idx].min() + 1e-8
        )

        # Hybrid score
        score = alpha * norm_typicality + (1 - alpha) * norm_distance
        best = cluster_idx[np.argmax(score)]
        selected.append(best)

    return np.array(selected)


selected_indices = select_hybrid(proc_embeddings, typicality, cluster_labels, budget, alpha=0.2)
print(f"Selected indices: {selected_indices}")
print(f"Selected labels: {labels[selected_indices]}")
print(f"Classes covered: {len(np.unique(labels[selected_indices]))}")

Selected indices: [11148 39670  5922  8907 11196 24391 34846  3866 18610  6492]
Selected labels: [8 0 2 7 1 9 5 4 6 3]
Classes covered: 10


## 7. Evaluation (Linear Probe)

In [9]:
def evaluate_selection(selected_indices, embeddings, labels, test_embeddings, test_labels):
    """Train a linear probe on selected examples and evaluate on test set."""
    sel_emb = embeddings[selected_indices]
    sel_lab = labels[selected_indices]
    clf = LogisticRegression(max_iter=1000)
    clf.fit(sel_emb, sel_lab)
    preds = clf.predict(test_embeddings)
    accuracy = accuracy_score(test_labels, preds)
    n_classes = len(np.unique(sel_lab))
    return {'accuracy': accuracy, 'n_classes_covered': n_classes}


results = evaluate_selection(selected_indices, proc_embeddings, labels, proc_test, test_labels)
print(f"\n=== Modified TypiClust Results (UMAP-5 + Hybrid alpha=0.2) ===")
print(f"Accuracy: {results['accuracy']:.4f}")
print(f"Classes covered: {results['n_classes_covered']}/10")


=== Modified TypiClust Results (UMAP-5 + Hybrid alpha=0.2) ===
Accuracy: 0.6949
Classes covered: 10/10


## 8. Comparison with Baseline

In [10]:
# Random baseline (30 seeds) on UMAP-reduced embeddings
random_accs = []
for seed in range(30):
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(proc_embeddings), size=budget, replace=False)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(proc_embeddings[idx], labels[idx])
    preds = clf.predict(proc_test)
    random_accs.append(accuracy_score(test_labels, preds))

random_accs = np.array(random_accs)

# Original TypiClust baseline (no UMAP, no hybrid) on raw embeddings
from sklearn.cluster import KMeans as KM
km = KM(n_clusters=budget, random_state=42, n_init=10)
km.fit(embeddings)
baseline_typ = compute_typicality(embeddings, k=20)
baseline_sel = []
for c in range(budget):
    cidx = np.where(km.labels_ == c)[0]
    baseline_sel.append(cidx[np.argmax(baseline_typ[cidx])])
baseline_sel = np.array(baseline_sel)
clf = LogisticRegression(max_iter=1000)
clf.fit(embeddings[baseline_sel], labels[baseline_sel])
baseline_acc = accuracy_score(test_labels, clf.predict(test_embeddings))

print(f"Random Baseline: {random_accs.mean():.4f} +/- {random_accs.std():.4f}")
print(f"Original TypiClust: {baseline_acc:.4f}")
print(f"Modified (UMAP-5 + Hybrid): {results['accuracy']:.4f}")
print(f"Improvement over baseline: +{results['accuracy'] - baseline_acc:.4f}")

Random Baseline: 0.4448 +/- 0.0722
Original TypiClust: 0.4476
Modified (UMAP-5 + Hybrid): 0.6949
Improvement over baseline: +0.2473
